# 4. Evals with DeepEval

**Time: about 20 minutes**

An **eval** runs the real agent on a real request, then scores what it did. A score below the threshold fails, like a test.

DeepEval gives us three pieces:

1. **A test case** (`LLMTestCase`): the request, the answer, the tools the agent called, and the tools we expected.
2. **A metric**: something that scores a test case between 0 and 1, with a reason.
3. **A threshold**: the lowest score that still passes.

There are two kinds of metrics:

| Kind | How it scores | Cost | Same result every run? | Example |
|---|---|---|---|---|
| Code metric | plain Python | free | yes | Tool Correctness |
| LLM judge | another model reads the test case and scores it | cents | mostly | Argument Correctness, Task Completion, GEval |

## Build a test case from a real run

In [1]:
from agent import run_agent
from evals.harness import to_test_case

case = {
    "id": "vegetarian-pizza",
    "request": "Order me a vegetarian pizza from a place that is open now.",
    "expected_tools": ["search_restaurants", "get_menu", "place_order"],
}
result = run_agent(case["request"])
test_case = to_test_case(case, result)

print("input:         ", test_case.input)
print("actual_output: ", test_case.actual_output)
print("tools_called:  ", [(t.name, t.input_parameters) for t in test_case.tools_called])
print("expected_tools:", [t.name for t in test_case.expected_tools])

input:          Order me a vegetarian pizza from a place that is open now.
actual_output:  I have ordered a Margherita Pizza, which is vegetarian, from Napoli Express for you.
tools_called:   [('search_restaurants', {'cuisine': 'pizza'}), ('get_menu', {'restaurant': 'Napoli Express'}), ('place_order', {'restaurant': 'Napoli Express', 'item': 'Margherita Pizza'})]
expected_tools: ['search_restaurants', 'get_menu', 'place_order']


A small helper to run a metric and print what it says. In a notebook we use `async_mode=False` so each metric runs in order.

In [2]:
def score(metric, test_case):
    metric.measure(test_case, _show_indicator=False)  # _show_indicator=False hides the progress bar
    verdict = "PASS" if metric.is_successful() else "FAIL"
    print(f"{metric.__name__}: {metric.score:.2f} (threshold {metric.threshold}) {verdict}")
    print("Reason:", metric.reason)

## Tool Correctness (code)

Did the agent call the tools we expected, in the right order? No LLM involved: it compares two lists.

In [3]:
from deepeval.metrics import ToolCorrectnessMetric

score(ToolCorrectnessMetric(threshold=1.0, should_consider_ordering=True, async_mode=False), test_case)

Tool Correctness: 1.00 (threshold 1.0) PASS
Reason: [
	 Tool Calling Reason: Correct ordering: all expected tools ['search_restaurants', 'get_menu', 'place_order'] were called in the correct order.
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]



Now a bad run. Here the agent placed the order **before** reading the menu:

In [4]:
from deepeval.test_case import LLMTestCase, ToolCall

skipped_the_menu = LLMTestCase(
    input=case["request"],
    actual_output="I ordered a Margherita Pizza from Napoli Express.",
    tools_called=[ToolCall(name="search_restaurants"), ToolCall(name="place_order"), ToolCall(name="get_menu")],
    expected_tools=[ToolCall(name=n) for n in case["expected_tools"]],
)
score(ToolCorrectnessMetric(threshold=1.0, should_consider_ordering=True, async_mode=False), skipped_the_menu)

Tool Correctness: 0.67 (threshold 1.0) FAIL
Reason: [
	 Tool Calling Reason: Incorrect tool usage: out-of-order tools ['place_order']; expected ['search_restaurants', 'get_menu', 'place_order'], called ['search_restaurants', 'place_order', 'get_menu']. See more details above.
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]



## Argument Correctness (LLM judge)

The right tools with the wrong arguments is still wrong. Checking "is this a sensible argument for this request" needs judgment,
so DeepEval asks a judge model. We pin the judge in `evals/harness.py`:

In [5]:
from deepeval.metrics import ArgumentCorrectnessMetric

from evals.harness import JUDGE_MODEL

print("Judge model:", JUDGE_MODEL)
score(ArgumentCorrectnessMetric(threshold=0.8, model=JUDGE_MODEL, async_mode=False), test_case)

Judge model: gpt-5.4


Argument Correctness: 1.00 (threshold 0.8) PASS
Reason: The score is 1.00 because the argument selection fully matches the request, with no incorrect tool calls and a clear fit to ordering a vegetarian pizza from a place that is open now.


And a bad run: the customer asked for a vegetarian pizza, and the agent ordered pepperoni.

In [6]:
pepperoni = LLMTestCase(
    input=case["request"],
    actual_output="I ordered a Pepperoni Pizza from Napoli Express.",
    tools_called=[
        ToolCall(name="search_restaurants", input_parameters={"cuisine": "pizza"}),
        ToolCall(name="get_menu", input_parameters={"restaurant": "Napoli Express"}),
        ToolCall(name="place_order", input_parameters={"restaurant": "Napoli Express", "item": "Pepperoni Pizza"}),
    ],
)
score(ArgumentCorrectnessMetric(threshold=0.8, model=JUDGE_MODEL, async_mode=False), pepperoni)

Argument Correctness: 0.67 (threshold 0.8) FAIL
Reason: The score is 0.67 because the request was understood as a pizza order from a place open now, but it is not higher because the selected item was Pepperoni Pizza, which does not satisfy the user's requirement for a vegetarian pizza.


## Task Completion (LLM judge)

Did the agent actually do what the customer asked? The judge reads the request, the tool calls and the answer.

In [7]:
from deepeval.metrics import TaskCompletionMetric

score(TaskCompletionMetric(threshold=0.7, model=JUDGE_MODEL, async_mode=False), test_case)

Task Completion: 0.96 (threshold 0.7) PASS
Reason: The actual outcome strongly matches the task: a vegetarian pizza (Margherita) was ordered from a specific restaurant, and the order appears completed as evidenced by the order ID. The only minor gap is that the outcome does not explicitly confirm the restaurant was currently open, though this is reasonably implied by the successful order.


In [8]:
asked_instead = LLMTestCase(
    input=case["request"],
    actual_output="Napoli Express has a Margherita Pizza for $12.50. Would you like me to order it?",
    tools_called=[ToolCall(name="search_restaurants"), ToolCall(name="get_menu")],
)
score(TaskCompletionMetric(threshold=0.7, model=JUDGE_MODEL, async_mode=False), asked_instead)

Task Completion: 0.72 (threshold 0.7) PASS
Reason: The actual outcome substantially supports the task by finding a currently open restaurant and identifying a vegetarian pizza option available to order. However, it did not complete the key action of actually ordering the pizza; it only prepared the order and asked for confirmation. Therefore, it partially but not fully achieved the desired outcome.


Look closely at that score. No order was placed, so the task was **not** done. But the judge gives partial credit
for finding the right pizza, so the score often lands close to the threshold, and it may even pass.
When we ran this while writing the class, it scored 0.72 and passed.

This is why we never rely on one metric. The next one, Answer Quality, was written for exactly this mistake.

## GEval: your own criteria, in plain English

The built in metrics cover common checks. For your product's own rules, write the criteria yourself with **GEval**.
Ours lives in `evals/harness.py`:

In [9]:
import inspect

from IPython.display import Code

from evals import harness

Code(inspect.getsource(harness.answer_quality), language="python")

def answer_quality() -> GEval:
    """A judge with our own criteria, written in plain English."""
    return GEval(
        name="Answer Quality",
        criteria=(
            "The reply tells the customer clearly what happened: what was ordered and where, "
            "or the status of their order, or why it could not be done. "
            "Every fact in the reply must match the tool results. "
            "It is one or two short sentences. "
            "It must not ask permission before acting (for example 'Shall I place the order?'), "
            "but suggesting a next step is fine."
        ),
        # Give the judge the evidence. Without the tool calls it cannot know the restaurant names are real.
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.TOOLS_CALLED],
        model=JUDGE_MODEL,
        threshold=0.7,
    )

In [10]:
def answer_quality():
    metric = harness.answer_quality()
    metric.async_mode = False
    return metric


score(answer_quality(), test_case)
print()
score(answer_quality(), asked_instead)

Answer Quality [GEval]: 1.00 (threshold 0.7) PASS
Reason: The response fully matches the request by selecting an open pizza place from the tool results, confirming Napoli Express was open and choosing the vegetarian Margherita Pizza from its menu before placing the order. Its factual claims align with the tool outputs, it clearly states the order outcome, and it stays within one short sentence without asking for permission.



Answer Quality [GEval]: 0.18 (threshold 0.7) FAIL
Reason: The response partially addresses the request by identifying an open restaurant and a vegetarian option, but it does not state that an order was actually placed or explain why it could not be completed. It also asks for permission before ordering, which conflicts with the instruction to avoid permission-seeking before acting. While concise, it relies on search_restaurants and get_menu only, so there is no tool evidence that an order was made, making the offer to order incomplete relative to the user's request.


## Choosing thresholds

* **Code metrics** are exact, so use a strict threshold. Tool Correctness is `1.0`: the order is either right or not.
* **LLM judges** wobble a little between runs. Run them on a few answers you know are good, look at the scores,
  and set the threshold a bit below. We use `0.8` for arguments and `0.7` for task completion and answer quality.
* If a threshold fails good answers, it is too strict. If it passes bad ones, it is too loose. Test it on both,
  like we just did.

## The judge is a model too

* It **costs money**: every judge metric is one or more API calls.
* It **can be wrong**: you will see this in notebook 5.
* It **can change**: pin the judge model, or your scores will move when the agent did not.

In [11]:
metric = TaskCompletionMetric(threshold=0.7, model=JUDGE_MODEL, async_mode=False)
metric.measure(test_case, _show_indicator=False)
print(f"That one judgment cost ${metric.evaluation_cost:.4f}")

That one judgment cost $0.0053


## Try it yourself

1. Write a test case where the agent orders from Slice House (closed). Which metrics catch it? Which do not?
2. Write a GEval called "Friendly Tone" and try it on a rude answer and a polite one.
3. Set `should_consider_ordering=False` on Tool Correctness and score `skipped_the_menu` again. When would you want that?